# TP - Perceptrón Multicapa (MLP) para Clasificación de Texto
### Redes Neuronales

Dataset: **20 Newsgroups** (~18.000 posts de foros de Usenet, clasificados en 20 categorías).

Este trabajo continúa el TP de **Regresión Softmax**: reutilizamos el pipeline de preprocesamiento, la vectorización TF-IDF, el tracking con TensorBoard / MLflow y la búsqueda con Optuna, pero ahora el modelo pasa a ser un **MLP** (capas ocultas + no-linealidades).

## Objetivos
- Extender la clase `SoftmaxRegression` para que acepte **capas ocultas** (1, 2 o 3), con distinta cantidad de neuronas y distintas funciones de activación.
- Buscar con **Optuna** (con **pruning**) la mejor combinación de hiperparámetros de arquitectura, de optimizador (Adam, RMSprop, SGD, con sus momentos de 1.º y 2.º orden) y los ya buscados en el TP anterior (preprocesamiento, `lr`, `batch_size`).
- Encontrar una combinación de capas y unidades que deje al problema **sobreajustado** (overfitting).
- Analizar los resultados a lo ancho de todos los entrenamientos (performance y tiempos) y **comparar contra la regresión softmax**

## 0. Setup

In [1]:
import re
import time
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from torch.utils.tensorboard import SummaryWriter
import mlflow
import mlflow.pytorch
import optuna

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("mlp_20newsgroups")

/Users/emanuel/Documents/MLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Usando dispositivo: cpu


2026/10/21 23:18:49 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/21 23:18:49 INFO mlflow.store.db.utils: Updating database tables
2026/10/21 23:18:53 INFO mlflow.tracking.fluent: Experiment with name 'mlp_20newsgroups' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/emanuel/Documents/MLP/mlruns/1', creation_time=1792635533564, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792635533564, lifecycle_stage='active', name='mlp_20newsgroups', tags={}, trace_location=None, workspace='default'>

In [2]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/emanuel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/emanuel/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/emanuel/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /Users/emanuel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/emanuel/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/emanuel/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 1. Datos: 20 Newsgroups

Igual que en el TP anterior, usamos `remove=('headers', 'footers', 'quotes')`.

In [3]:
newsgroups_train = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
newsgroups_test  = fetch_20newsgroups(subset="test",  remove=("headers", "footers", "quotes"))

class_names = newsgroups_train.target_names
NUM_CLASSES = len(class_names)
print(f"Clases ({NUM_CLASSES}):", class_names)
print(f"\nTrain: {len(newsgroups_train.data)} documentos | Test: {len(newsgroups_test.data)} documentos")

print("\n--- Ejemplo de documento ---")
print("Categoría:", class_names[newsgroups_train.target[0]])
print(newsgroups_train.data[0][:500])

Clases (20): ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']

Train: 11314 documentos | Test: 7532 documentos

--- Ejemplo de documento ---
Categoría: rec.autos
I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


## 2. Preprocesamiento configurable

Misma función del TP anterior (ya resuelta). Hiperparámetros: `remove_stopwords` y `normalization` (`"none"`, `"stem"`, `"lemmatize"`).

In [4]:
STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()
LEMMATIZER = WordNetLemmatizer()


def preprocess_text(text, remove_stopwords, normalization):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]

    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP_WORDS]

    if normalization == "stem":
        tokens = [STEMMER.stem(t) for t in tokens]
    elif normalization == "lemmatize":
        tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    # si normalization == "none", no se hace nada más

    return " ".join(tokens)

In [5]:
# Sanity check: probamos las distintas combinaciones sobre una oración de ejemplo
ejemplo = "The researchers were studying the running processes and stopping conditions of the engines."
for norm in ["none", "stem", "lemmatize"]:
    for rm_sw in [False, True]:
        resultado = preprocess_text(ejemplo, remove_stopwords=rm_sw, normalization=norm)
        print(f"normalization={norm:10s} remove_stopwords={str(rm_sw):5s} -> {resultado}")

normalization=none       remove_stopwords=False -> the researchers were studying the running processes and stopping conditions of the engines
normalization=none       remove_stopwords=True  -> researchers studying running processes stopping conditions engines
normalization=stem       remove_stopwords=False -> the research were studi the run process and stop condit of the engin
normalization=stem       remove_stopwords=True  -> research studi run process stop condit engin
normalization=lemmatize  remove_stopwords=False -> the researcher were studying the running process and stopping condition of the engine
normalization=lemmatize  remove_stopwords=True  -> researcher studying running process stopping condition engine


## 3. Vectorización TF-IDF

Hiperparámetros: `ngram_range`, `max_features`, `min_df`.

In [6]:
def vectorize_data(train_texts, test_texts, ngram_range, max_features, min_df):
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features, min_df=min_df)
    X_train = vectorizer.fit_transform(train_texts)   # el vocabulario se arma SOLO con train
    X_test = vectorizer.transform(test_texts)
    return X_train, X_test, vectorizer

In [7]:
# Sanity check rápido con pocos documentos
_sample_train = ["the cat sat on the mat", "dogs are running in the park", "cats and dogs are pets"]
_sample_test = ["the dog sat on the mat"]
_Xtr, _Xte, _vec = vectorize_data(_sample_train, _sample_test, ngram_range=(1, 1), max_features=50, min_df=2)
print("Shape train:", _Xtr.shape, "| Shape test:", _Xte.shape)
print("Vocabulario:", sorted(_vec.vocabulary_.keys()))
assert set(_vec.vocabulary_) == {"the", "are", "dogs"}, "Revisá que vectorize_data use los parámetros recibidos"

Shape train: (3, 3) | Shape test: (1, 3)
Vocabulario: ['are', 'dogs', 'the']


## 4. Modelo: MLP (generalización de la regresión softmax)

Extendemos la clase del TP anterior para que reciba una **lista de tamaños de capas ocultas** (`hidden_units`) y una **función de activación**:

```
x → [Linear → Activación → (Dropout)] × n_capas_ocultas → Linear → logits
```

- Con `hidden_units=()` no hay capas ocultas y el modelo es exactamente la regresión softmax del TP anterior (una única `nn.Linear`). Esto nos permite usar la misma clase para el baseline.
- Igual que antes, el `forward` devuelve **logits crudos** (sin softmax), porque `CrossEntropyLoss` ya aplica `log_softmax`.
- `Dropout` es opcional (`dropout=0.0` lo desactiva). Lo incluimos como hiperparámetro de regularización, ya que un MLP sobreajusta con mucha facilidad sobre TF-IDF.

In [8]:
ACTIVATIONS = {
    "relu": lambda: nn.ReLU(),
    "tanh": lambda: nn.Tanh(),
    "leaky_relu": lambda: nn.LeakyReLU(0.01),
    "gelu": lambda: nn.GELU(),
    "elu": lambda: nn.ELU(),
}


class MLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_units=(), activation="relu", dropout=0.0):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_units:
            layers.append(nn.Linear(prev, h))
            layers.append(ACTIVATIONS[activation]())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        self.hidden = nn.Sequential(*layers)         
        self.out = nn.Linear(prev, num_classes)        

    def forward(self, x):
        return self.out(self.hidden(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [9]:
# Sanity checks
_dummy = torch.randn(4, 50)

_softmax = MLP(50, NUM_CLASSES, hidden_units=())
assert len(_softmax.hidden) == 0 and _softmax(_dummy).shape == (4, NUM_CLASSES)
assert count_params(_softmax) == 50 * NUM_CLASSES + NUM_CLASSES 
print("Sin capas ocultas  (= regresión softmax):", count_params(_softmax), "parámetros")

for act in ACTIVATIONS:
    m = MLP(50, NUM_CLASSES, hidden_units=(64, 32, 16), activation=act, dropout=0.3)
    assert m(_dummy).shape == (4, NUM_CLASSES)
print("3 capas ocultas (64, 32, 16):", count_params(m), "parámetros | forward OK para todas las activaciones")

Sin capas ocultas  (= regresión softmax): 1020 parámetros
3 capas ocultas (64, 32, 16): 6212 parámetros | forward OK para todas las activaciones


## 5. Entrenamiento

Mismo esquema que el TP anterior, con tres agregados:

1. **Optimizador configurable** (`build_optimizer`): Adam (`beta1`, `beta2` = momentos de 1.º y 2.º orden), RMSprop (`alpha` = decaimiento del momento de 2.º orden, `momentum`) y SGD (`momentum` = momento de 1.º orden, Nesterov opcional). Todos con `weight_decay`.
2. **Pruning**: si se pasa un `trial` de Optuna, en cada época se reporta la accuracy de validación (`trial.report`) y, si el pruner decide que el trial no promete, se corta con `TrialPruned`.
3. **Early stopping** (`patience`) y **medición de tiempos** (por época y total).

> Los tiempos incluyen el entrenamiento y la evaluación sobre validación de cada época (en GPU se sincroniza antes de medir).

In [10]:
def to_tensor_dataset(X, y):
    X_dense = torch.tensor(X.toarray(), dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    return TensorDataset(X_dense, y_tensor)


def make_loaders(X_train, y_train, X_val, y_val, batch_size):
    train_loader = DataLoader(to_tensor_dataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(to_tensor_dataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader


def build_optimizer(model, p):
    """Construye el optimizador a partir del dict de hiperparámetros `p`."""
    name, wd = p["optimizer"], p["weight_decay"]
    if name == "adam":
        return optim.Adam(model.parameters(), lr=p["lr_adam"],
                          betas=(p["beta1"], p["beta2"]), weight_decay=wd)
    if name == "rmsprop":
        return optim.RMSprop(model.parameters(), lr=p["lr_rmsprop"], alpha=p["rms_alpha"],
                             momentum=p["rms_momentum"], weight_decay=wd)
    if name == "sgd":
        nesterov = bool(p["sgd_nesterov"]) and p["sgd_momentum"] > 0   # Nesterov exige momentum > 0
        return optim.SGD(model.parameters(), lr=p["lr_sgd"], momentum=p["sgd_momentum"],
                         nesterov=nesterov, weight_decay=wd)
    raise ValueError(f"Optimizador desconocido: {name}")


def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


def train_model(model, train_loader, val_loader, optimizer, epochs, writer=None, run_name="model",
                trial=None, patience=None, verbose=True):
    """Entrena el modelo y devuelve el historial (métricas por época + resumen).

    - trial: si es un trial de Optuna, se reporta la val_acc por época y se aplica pruning.
    - patience: si no es None, early stopping sobre val_acc.
    """
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "epoch_time": []}
    best_val_acc, best_epoch, epochs_no_improve = -1.0, 0, 0
    pruned = False
    log_to_mlflow = mlflow.active_run() is not None

    for epoch in range(epochs):
        _sync()
        t0 = time.perf_counter()

        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                running_loss += loss.item() * xb.size(0)
                correct += (logits.argmax(dim=1) == yb).sum().item()
                total += xb.size(0)
        val_loss, val_acc = running_loss / total, correct / total

        _sync()
        history["epoch_time"].append(time.perf_counter() - t0)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if verbose:
            print(f"[{run_name}] Epoch {epoch+1}/{epochs} - "
                  f"train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} - "
                  f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

        if writer is not None:
            writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
            writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
        if log_to_mlflow:
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("train_acc", train_acc, step=epoch)
            mlflow.log_metric("val_acc", val_acc, step=epoch)

        if val_acc > best_val_acc:
            best_val_acc, best_epoch, epochs_no_improve = val_acc, epoch + 1, 0
        else:
            epochs_no_improve += 1

        # Pruning (Optuna): reportamos el valor intermedio y consultamos al pruner
        if trial is not None:
            trial.report(val_acc, epoch)
            if trial.should_prune():
                pruned = True
                break

        # Early stopping
        if patience is not None and epochs_no_improve >= patience:
            break

    history["train_time"] = float(sum(history["epoch_time"]))
    history["epochs_run"] = len(history["train_loss"])
    history["best_val_acc"] = best_val_acc
    history["best_epoch"] = best_epoch
    history["final_train_acc"] = history["train_acc"][-1]
    history["final_val_acc"] = history["val_acc"][-1]

    if trial is not None:
        for k in ["train_time", "epochs_run", "best_epoch", "final_train_acc", "final_val_acc"]:
            trial.set_user_attr(k, history[k])
    if pruned:
        raise optuna.TrialPruned()
    return history


def evaluate_model(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            preds.extend(logits.argmax(dim=1).cpu().numpy())
            labels.extend(yb.numpy())
    return np.array(preds), np.array(labels)

In [11]:
%load_ext tensorboard

## 6. Búsqueda de hiperparámetros con Optuna (con pruning)

### Espacio de búsqueda

| Grupo | Hiperparámetro | Valores |
|---|---|---|
| Preprocesamiento | `remove_stopwords`, `normalization`, `ngram_range`, `max_features`, `min_df` | igual que el TP anterior |
| Entrenamiento | `batch_size` | 32, 64, 128, 256 |
| Arquitectura | `n_layers` | 1, 2, 3 |
| | `units_l1 … units_lN` (neuronas por capa, **independientes** para cada capa) | 32, 64, 128, 256, 512, 1024 |
| | `activation` | relu, tanh, leaky_relu, gelu, elu |
| | `dropout` | 0.0 a 0.6 |
| Optimizador | `optimizer` | adam, rmsprop, sgd |
| | `weight_decay` | 1e-8 a 1e-3 (log) |
| | Adam: `lr_adam`, `beta1` (1.er momento), `beta2` (2.º momento) | 1e-4–1e-2, 0.8–0.99, 0.9–0.9999 |
| | RMSprop: `lr_rmsprop`, `rms_alpha` (2.º momento), `rms_momentum` (1.er momento) | 1e-4–1e-2, 0.9–0.999, 0–0.9 |
| | SGD: `lr_sgd`, `sgd_momentum` (1.er momento), `sgd_nesterov` | 1e-2–10, 0–0.99, True/False |

Los parámetros de cada optimizador son **condicionales**: Optuna solo los muestrea cuando elige ese optimizador. El `lr` tiene un nombre distinto por optimizador (`lr_adam`, `lr_rmsprop`, `lr_sgd`) porque sus rangos razonables son muy distintos (SGD con vectores TF-IDF necesita learning rates mucho mayores).

### Pruning
Usamos `MedianPruner`: en cada época se compara la accuracy de validación del trial contra la mediana de los trials anteriores en esa misma época y se corta si está por debajo. Con `n_startup_trials=8` los primeros trials nunca se podan (para tener una referencia) y con `n_warmup_steps=3` ningún trial se poda en sus primeras 4 épocas.

### Optimizaciones de tiempo
- El preprocesamiento (lo más lento) depende solo de `remove_stopwords` y `normalization`: se precalcula una vez para las 6 combinaciones.
- La vectorización TF-IDF se cachea por combinación de hiperparámetros, así que si Optuna repite una combinación no se recalcula.
- La partición train/validación (80/20 estratificada) es **fija** para todos los trials, así se comparan sobre los mismos datos. El test **no** se usa en la búsqueda.

In [ ]:
print("Precalculando preprocesamiento para las 6 combinaciones posibles (puede tardar unos minutos)...")
preprocessed_cache = {}
for remove_stopwords in [False, True]:
    for normalization in ["none", "stem", "lemmatize"]:
        key = (remove_stopwords, normalization)
        train_proc = [preprocess_text(t, remove_stopwords, normalization) for t in newsgroups_train.data]
        test_proc  = [preprocess_text(t, remove_stopwords, normalization) for t in newsgroups_test.data]
        preprocessed_cache[key] = (train_proc, test_proc)
        print("  listo:", key)

Precalculando preprocesamiento para las 6 combinaciones posibles (puede tardar unos minutos)...
  listo: (False, 'none')
  listo: (False, 'stem')
  listo: (False, 'lemmatize')
  listo: (True, 'none')


In [ ]:
# Partición fija train/validación para la búsqueda
y_all = np.array(newsgroups_train.target)
SEARCH_IDX_TR, SEARCH_IDX_VAL = train_test_split(
    np.arange(len(y_all)), test_size=0.2, random_state=SEED, stratify=y_all
)
y_search_tr, y_search_val = y_all[SEARCH_IDX_TR], y_all[SEARCH_IDX_VAL]

_features_cache = {}


def ngram_tuple(name):
    return (1, 1) if name == "1gram" else (1, 2)


def get_search_features(remove_stopwords, normalization, ngram_range, max_features, min_df):
    """Matrices TF-IDF (esparsas) de la partición de búsqueda, con caché."""
    key = (remove_stopwords, normalization, ngram_range, max_features, min_df)
    if key not in _features_cache:
        train_texts, _ = preprocessed_cache[(remove_stopwords, normalization)]
        tr_texts = [train_texts[i] for i in SEARCH_IDX_TR]
        val_texts = [train_texts[i] for i in SEARCH_IDX_VAL]
        X_tr, X_val, _ = vectorize_data(tr_texts, val_texts, ngram_range, max_features, min_df)
        _features_cache[key] = (X_tr, X_val)
    return _features_cache[key]

In [ ]:
ACTIVATION_NAMES = list(ACTIVATIONS.keys())
OPTIMIZER_NAMES = ["adam", "rmsprop", "sgd"]


def suggest_params(trial, use_hidden=True):
    """Muestrea todos los hiperparámetros y los devuelve en un dict.
    use_hidden=False -> baseline softmax (sin hiperparámetros de arquitectura)."""
    p = {}
    # --- preprocesamiento (igual que el TP anterior) ---
    p["remove_stopwords"] = trial.suggest_categorical("remove_stopwords", [False, True])
    p["normalization"] = trial.suggest_categorical("normalization", ["none", "stem", "lemmatize"])
    p["ngram_range"] = trial.suggest_categorical("ngram_range", ["1gram", "1_2gram"])
    p["max_features"] = trial.suggest_categorical("max_features", [2000, 5000, 10000])
    p["min_df"] = trial.suggest_categorical("min_df", [1, 2, 5])
    p["batch_size"] = trial.suggest_categorical("batch_size", [32, 64, 128, 256])

    # --- arquitectura ---
    if use_hidden:
        p["n_layers"] = trial.suggest_int("n_layers", 1, 3)
        for i in range(p["n_layers"]):
            p[f"units_l{i+1}"] = trial.suggest_categorical(f"units_l{i+1}", [32, 64, 128, 256, 512, 1024])
        p["activation"] = trial.suggest_categorical("activation", ACTIVATION_NAMES)
        p["dropout"] = trial.suggest_float("dropout", 0.0, 0.6, step=0.1)

    # --- optimizador ---
    p["optimizer"] = trial.suggest_categorical("optimizer", OPTIMIZER_NAMES)
    p["weight_decay"] = trial.suggest_float("weight_decay", 1e-8, 1e-3, log=True)
    if p["optimizer"] == "adam":
        p["lr_adam"] = trial.suggest_float("lr_adam", 1e-4, 1e-2, log=True)
        p["beta1"] = trial.suggest_float("beta1", 0.8, 0.99)       # momento de 1.er orden
        p["beta2"] = trial.suggest_float("beta2", 0.9, 0.9999)     # momento de 2.º orden
    elif p["optimizer"] == "rmsprop":
        p["lr_rmsprop"] = trial.suggest_float("lr_rmsprop", 1e-4, 1e-2, log=True)
        p["rms_alpha"] = trial.suggest_float("rms_alpha", 0.9, 0.999)       # momento de 2.º orden
        p["rms_momentum"] = trial.suggest_float("rms_momentum", 0.0, 0.9)   # momento de 1.er orden
    else:
        p["lr_sgd"] = trial.suggest_float("lr_sgd", 1e-2, 10.0, log=True)
        p["sgd_momentum"] = trial.suggest_float("sgd_momentum", 0.0, 0.99)  # momento de 1.er orden
        p["sgd_nesterov"] = trial.suggest_categorical("sgd_nesterov", [False, True])
    return p


def hidden_units_from(p):
    return tuple(p[f"units_l{i+1}"] for i in range(p.get("n_layers", 0)))


def build_model(p, input_dim, use_hidden=True):
    if use_hidden:
        return MLP(input_dim, NUM_CLASSES, hidden_units_from(p), p["activation"], p["dropout"])
    return MLP(input_dim, NUM_CLASSES, hidden_units=())    # regresión softmax

In [ ]:
EPOCHS_SEARCH = 20      # máximo de épocas por trial
PATIENCE = 5            # early stopping dentro de cada trial


def make_objective(use_hidden, tag):
    def objective(trial):
        p = suggest_params(trial, use_hidden)
        set_seed(SEED)

        X_tr, X_val = get_search_features(p["remove_stopwords"], p["normalization"],
                                          ngram_tuple(p["ngram_range"]), p["max_features"], p["min_df"])
        train_loader, val_loader = make_loaders(X_tr, y_search_tr, X_val, y_search_val, p["batch_size"])

        model = build_model(p, X_tr.shape[1], use_hidden)
        trial.set_user_attr("n_params", count_params(model))
        optimizer = build_optimizer(model, p)
        writer = SummaryWriter(log_dir=f"runs/{tag}/trial_{trial.number:03d}")

        pruned = False
        with mlflow.start_run(run_name=f"{tag}_trial_{trial.number:03d}", nested=True):
            mlflow.log_params({**p, "n_params": count_params(model)})
            try:
                hist = train_model(model, train_loader, val_loader, optimizer, EPOCHS_SEARCH,
                                   writer=writer, run_name=f"{tag}_{trial.number}",
                                   trial=trial, patience=PATIENCE, verbose=False)
                mlflow.set_tag("optuna_state", "COMPLETE")
                mlflow.log_metrics({"best_val_acc": hist["best_val_acc"], "train_time_s": hist["train_time"]})
            except optuna.TrialPruned:
                pruned = True
                mlflow.set_tag("optuna_state", "PRUNED")
            finally:
                writer.close()

        if pruned:
            raise optuna.TrialPruned()
        # Valor a maximizar: mejor val_acc alcanzada (equivale a quedarse con el mejor checkpoint)
        return hist["best_val_acc"]
    return objective

In [ ]:
N_TRIALS_MLP = 40

study_mlp = optuna.create_study(
    direction="maximize",
    study_name="mlp_20newsgroups",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=3),
)

with mlflow.start_run(run_name="optuna_mlp_search"):
    study_mlp.optimize(make_objective(use_hidden=True, tag="mlp_search"), n_trials=N_TRIALS_MLP)
    mlflow.log_params({f"best_{k}": v for k, v in study_mlp.best_params.items()})
    mlflow.log_metric("best_val_acc", study_mlp.best_value)

print("\nMejores hiperparámetros:", study_mlp.best_params)
print("Mejor accuracy de validación:", round(study_mlp.best_value, 4))
print("Mejor época del mejor trial:", study_mlp.best_trial.user_attrs["best_epoch"])

## 7. Experimento de overfitting: ¿cuánta capacidad hace falta para sobreajustar?

La consigna pide encontrar una cantidad de capas y unidades ocultas con la que el problema quede **sobreajustado**. Para aislar el efecto de la capacidad:

- Fijamos todo lo demás: el preprocesamiento/vectorización del mejor trial, `Adam(lr=1e-3)`, activación ReLU, **sin dropout y sin weight decay** (sin regularización).
- Variamos la **profundidad** (1, 2, 3 capas ocultas) y el **ancho** (64, 256, 1024 neuronas por capa) en una grilla.
- Entrenamos un número fijo de épocas, **sin early stopping**, sobre la misma partición train/validación de la búsqueda.

Criterio: consideramos que el modelo "sobreajusta" cuando alcanza una accuracy de entrenamiento ≥ 0.99 mientras que la accuracy de validación queda muy por debajo (gran *gap*). De todas las configuraciones que cumplen, elegimos la de **menor cantidad de parámetros**.

In [ ]:
OVERFIT_EPOCHS = 30
OVERFIT_DEPTHS = [1, 2, 3]
OVERFIT_WIDTHS = [64, 256, 1024]
OVERFIT_TRAIN_ACC = 0.99

bp = study_mlp.best_params
X_tr_of, X_val_of = get_search_features(bp["remove_stopwords"], bp["normalization"],
                                        ngram_tuple(bp["ngram_range"]), bp["max_features"], bp["min_df"])
train_loader_of, val_loader_of = make_loaders(X_tr_of, y_search_tr, X_val_of, y_search_val, batch_size=64)

overfit_rows, overfit_hist = [], {}
for depth in OVERFIT_DEPTHS:
    for width in OVERFIT_WIDTHS:
        set_seed(SEED)
        model_of = MLP(X_tr_of.shape[1], NUM_CLASSES, hidden_units=(width,) * depth, activation="relu", dropout=0.0)
        opt_of = optim.Adam(model_of.parameters(), lr=1e-3)
        with mlflow.start_run(run_name=f"overfit_{depth}x{width}"):
            mlflow.log_params({"depth": depth, "width": width, "epochs": OVERFIT_EPOCHS, "lr": 1e-3,
                               "n_params": count_params(model_of)})
            h = train_model(model_of, train_loader_of, val_loader_of, opt_of, OVERFIT_EPOCHS,
                            run_name=f"{depth}x{width}", patience=None, verbose=False)
        overfit_hist[(depth, width)] = h
        reached = [i + 1 for i, a in enumerate(h["train_acc"]) if a >= OVERFIT_TRAIN_ACC]
        overfit_rows.append({
            "capas": depth, "neuronas/capa": width, "parámetros": count_params(model_of),
            "train_acc final": h["final_train_acc"], "val_acc final": h["final_val_acc"],
            "mejor val_acc": h["best_val_acc"], "mejor época": h["best_epoch"],
            "gap (train - val)": h["final_train_acc"] - h["final_val_acc"],
            "época con train≥0.99": reached[0] if reached else None,
            "val_loss mín → final": f"{min(h['val_loss']):.3f} → {h['val_loss'][-1]:.3f}",
            "tiempo (s)": h["train_time"],
        })
        print(f"{depth}x{width:<5d} train_acc={h['final_train_acc']:.4f} val_acc={h['final_val_acc']:.4f} "
              f"tiempo={h['train_time']:.1f}s")

df_overfit = pd.DataFrame(overfit_rows)
df_overfit.round(4)

In [ ]:
ok = df_overfit[df_overfit["train_acc final"] >= OVERFIT_TRAIN_ACC].sort_values("parámetros")
if len(ok):
    chosen = ok.iloc[0]
else:
    print(f"Ninguna configuración llegó a train_acc >= {OVERFIT_TRAIN_ACC}; se usa la de mayor train_acc.")
    chosen = df_overfit.sort_values("train_acc final", ascending=False).iloc[0]
c_depth, c_width = int(chosen["capas"]), int(chosen["neuronas/capa"])
print(f"Configuración más chica que sobreajusta: {c_depth} capa(s) oculta(s) de {c_width} neuronas "
      f"({int(chosen['parámetros']):,} parámetros)")
print(f"  train_acc={chosen['train_acc final']:.4f} | "
      f"val_acc={chosen['val_acc final']:.4f} | gap={chosen['gap (train - val)']:.4f}")

h = overfit_hist[(c_depth, c_width)]
epochs_axis = range(1, len(h["train_acc"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].plot(epochs_axis, h["train_acc"], label="train")
axes[0].plot(epochs_axis, h["val_acc"], label="validación")
axes[0].set(title=f"Accuracy - {c_depth}x{c_width}", xlabel="época", ylabel="accuracy")
axes[0].legend()

axes[1].plot(epochs_axis, h["train_loss"], label="train")
axes[1].plot(epochs_axis, h["val_loss"], label="validación")
axes[1].set(title=f"Loss - {c_depth}x{c_width}", xlabel="época", ylabel="cross-entropy")
axes[1].legend()

gap = df_overfit.pivot(index="capas", columns="neuronas/capa", values="gap (train - val)")
im = axes[2].imshow(gap.values, cmap="Reds", aspect="auto")
axes[2].set_xticks(range(len(gap.columns)), gap.columns)
axes[2].set_yticks(range(len(gap.index)), gap.index)
axes[2].set(title="Gap train - val (accuracy)", xlabel="neuronas por capa", ylabel="capas ocultas")
for i in range(gap.shape[0]):
    for j in range(gap.shape[1]):
        axes[2].text(j, i, f"{gap.values[i, j]:.2f}", ha="center", va="center")
plt.colorbar(im, ax=axes[2])
plt.tight_layout()
plt.show()

## 8. Entrenamiento final del MLP

Con los mejores hiperparámetros de Optuna entrenamos con **todo** el train set y evaluamos contra el test set real (que no se usó en la búsqueda).

La cantidad de épocas **no se elige mirando el test**: usamos la `best_epoch` (mejor época sobre validación) del mejor trial. Durante el entrenamiento se grafica la accuracy sobre test por época solo para monitoreo, y el número que se reporta es el del final.

In [ ]:
def run_final(params, use_hidden, tag, epochs):
    """Entrena con todo el train set y evalúa sobre el test set. Devuelve modelo, historial y predicciones."""
    train_texts, test_texts = preprocessed_cache[(params["remove_stopwords"], params["normalization"])]
    X_train, X_test, _ = vectorize_data(train_texts, test_texts, ngram_tuple(params["ngram_range"]),
                                        params["max_features"], params["min_df"])
    train_loader, test_loader = make_loaders(X_train, newsgroups_train.target,
                                             X_test, newsgroups_test.target, params["batch_size"])

    set_seed(SEED)
    model = build_model(params, X_train.shape[1], use_hidden)
    optimizer = build_optimizer(model, params)
    writer = SummaryWriter(log_dir=f"runs/{tag}_final")

    with mlflow.start_run(run_name=f"{tag}_final"):
        mlflow.log_params({**params, "epochs": epochs, "n_params": count_params(model)})
        history = train_model(model, train_loader, test_loader, optimizer, epochs,
                              writer=writer, run_name=tag, patience=None)   # "val" = test (solo monitoreo)
        preds, labels = evaluate_model(model, test_loader)
        mlflow.log_metrics({"test_accuracy": accuracy_score(labels, preds),
                            "test_macro_f1": f1_score(labels, preds, average="macro"),
                            "train_time_s": history["train_time"]})
        try:
            mlflow.pytorch.log_model(model, "model", serialization_format="pickle")
        except Exception as e:
            print("No se pudo loguear el modelo en MLflow:", e)
    writer.close()
    return {"model": model, "history": history, "preds": preds, "labels": labels,
            "params": params, "n_params": count_params(model)}

In [ ]:
mlflow.end_run()
EPOCHS_FINAL_MLP = study_mlp.best_trial.user_attrs["best_epoch"]
print("Mejores hiperparámetros MLP:", study_mlp.best_params)
print("Épocas del entrenamiento final:", EPOCHS_FINAL_MLP)

res_mlp = run_final(study_mlp.best_params, use_hidden=True, tag="mlp", epochs=EPOCHS_FINAL_MLP)

In [ ]:
%tensorboard --logdir runs

## 9. Evaluación final del MLP

In [ ]:
def plot_confusion(cm, title):
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, cmap="Blues")
    plt.colorbar()
    plt.xticks(range(NUM_CLASSES), class_names, rotation=90)
    plt.yticks(range(NUM_CLASSES), class_names)
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def top_confusions(cm, k=6):
    off = cm.copy()
    np.fill_diagonal(off, 0)
    rows = []
    for idx in np.argsort(off, axis=None)[::-1][:k]:
        i, j = np.unravel_index(idx, off.shape)
        rows.append((class_names[i], class_names[j], int(off[i, j])))
    return pd.DataFrame(rows, columns=["clase real", "predicha como", "cantidad"])


acc_mlp = accuracy_score(res_mlp["labels"], res_mlp["preds"])
print(f"Accuracy en test (MLP): {acc_mlp:.4f}\n")
print(classification_report(res_mlp["labels"], res_mlp["preds"], target_names=class_names))
cm_mlp = confusion_matrix(res_mlp["labels"], res_mlp["preds"])
plot_confusion(cm_mlp, "Matriz de confusión - MLP sobre 20 Newsgroups")
top_confusions(cm_mlp)

## 10. Baseline: Regresión Softmax en el mismo pipeline

Para que la comparación sea justa, volvemos a buscar hiperparámetros para la regresión softmax **con el mismo código, el mismo pruner y el mismo espacio de búsqueda** (preprocesamiento, `batch_size`, optimizador y sus parámetros), pero sin hiperparámetros de arquitectura: `use_hidden=False` construye `MLP(hidden_units=())`, que es exactamente una capa lineal.

In [ ]:
N_TRIALS_SOFTMAX = 20

study_softmax = optuna.create_study(
    direction="maximize",
    study_name="softmax_20newsgroups",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=3),
)

with mlflow.start_run(run_name="optuna_softmax_search"):
    study_softmax.optimize(make_objective(use_hidden=False, tag="softmax_search"), n_trials=N_TRIALS_SOFTMAX)
    mlflow.log_params({f"best_{k}": v for k, v in study_softmax.best_params.items()})
    mlflow.log_metric("best_val_acc", study_softmax.best_value)

print("\nMejores hiperparámetros (softmax):", study_softmax.best_params)
print("Mejor accuracy de validación:", round(study_softmax.best_value, 4))

In [ ]:
mlflow.end_run()
EPOCHS_FINAL_SOFTMAX = study_softmax.best_trial.user_attrs["best_epoch"]
res_softmax = run_final(study_softmax.best_params, use_hidden=False, tag="softmax", epochs=EPOCHS_FINAL_SOFTMAX)

In [ ]:
def summarize(name, res, study):
    h = res["history"]
    search_time = sum(t.duration.total_seconds() for t in study.trials if t.duration is not None)
    return {
        "modelo": name,
        "val_acc (búsqueda)": study.best_value,
        "test_acc": accuracy_score(res["labels"], res["preds"]),
        "test_macro_F1": f1_score(res["labels"], res["preds"], average="macro"),
        "train_acc final": h["final_train_acc"],
        "épocas finales": h["epochs_run"],
        "tiempo entrenamiento final (s)": h["train_time"],
        "s/época": float(np.mean(h["epoch_time"])),
        "parámetros": res["n_params"],
        "tiempo de búsqueda (s)": search_time,
        "trials (completos/podados)": f"{sum(t.state.name == 'COMPLETE' for t in study.trials)}/"
                                      f"{sum(t.state.name == 'PRUNED' for t in study.trials)}",
    }


df_compare = pd.DataFrame([summarize("Softmax", res_softmax, study_softmax),
                           summarize("MLP", res_mlp, study_mlp)]).set_index("modelo")
df_compare.T

In [ ]:
f1_soft = f1_score(res_softmax["labels"], res_softmax["preds"], average=None)
f1_mlp = f1_score(res_mlp["labels"], res_mlp["preds"], average=None)

fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))

# F1 por clase: MLP - Softmax
diff = f1_mlp - f1_soft
order = np.argsort(diff)
axes[0].barh(np.array(class_names)[order], diff[order], color=np.where(diff[order] >= 0, "tab:green", "tab:red"))
axes[0].axvline(0, color="k", lw=0.8)
axes[0].set(title="Diferencia de F1 por clase (MLP − Softmax)", xlabel="Δ F1")

# Accuracy sobre test por época (monitoreo)
for res, label in [(res_softmax, "Softmax"), (res_mlp, "MLP")]:
    h = res["history"]
    axes[1].plot(range(1, len(h["val_acc"]) + 1), h["val_acc"], marker="o", label=f"{label} (test)")
    axes[1].plot(range(1, len(h["train_acc"]) + 1), h["train_acc"], ls="--", label=f"{label} (train)")
axes[1].set(title="Accuracy por época en el entrenamiento final", xlabel="época", ylabel="accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()

cm_soft = confusion_matrix(res_softmax["labels"], res_softmax["preds"])
print("Confusiones más frecuentes - Softmax")
display(top_confusions(cm_soft))
print("Confusiones más frecuentes - MLP")
display(top_confusions(cm_mlp))

## 11. Análisis de todos los trials del MLP

Acá miramos **a lo ancho de toda la búsqueda** (no solo el mejor trial): cuántos trials se podaron y cuánto tiempo se ahorró, cómo rinden y cuánto tardan los distintos optimizadores, activaciones y profundidades, y cómo se relaciona el rendimiento con el tamaño del modelo.

> Los gráficos y promedios por grupo usan solo trials **completos** (los podados no tienen un valor final comparable). Con pocos trials por categoría, las diferencias pueden ser ruido: fijarse en `n` y en la dispersión, no solo en el promedio.

In [ ]:
def trials_df(study):
    df = study.trials_dataframe(attrs=("number", "value", "state", "duration", "params", "user_attrs"))
    df["duration_s"] = df["duration"].dt.total_seconds()
    return df


df_mlp = trials_df(study_mlp)
done = df_mlp[df_mlp["state"] == "COMPLETE"].copy()
prun = df_mlp[df_mlp["state"] == "PRUNED"].copy()

print(f"Trials: {len(df_mlp)} | completos: {len(done)} | podados: {len(prun)}")
if len(prun) and len(done):
    est_full = len(prun) * done["user_attrs_train_time"].mean()
    real = prun["user_attrs_train_time"].sum()
    print(f"Tiempo de entrenamiento gastado en trials podados: {real:.0f} s")
    print(f"Estimación si no se hubiera podado (mismo tiempo medio que un trial completo): {est_full:.0f} s "
          f"→ ahorro aprox. {est_full - real:.0f} s ({100 * (1 - real / est_full):.0f}%)")
    print(f"Épocas promedio: completos {done['user_attrs_epochs_run'].mean():.1f} | podados {prun['user_attrs_epochs_run'].mean():.1f}")

print("\nTop 10 trials:")
cols = ["number", "value", "user_attrs_best_epoch", "user_attrs_train_time", "user_attrs_n_params",
        "params_n_layers", "params_activation", "params_optimizer", "params_dropout", "params_weight_decay",
        "params_batch_size", "params_normalization", "params_ngram_range", "params_max_features"]
done.sort_values("value", ascending=False)[cols].head(10).round(4)

In [ ]:
def summary_by(col):
    g = done.groupby(col).agg(
        n=("value", "count"),
        val_acc_media=("value", "mean"),
        val_acc_max=("value", "max"),
        tiempo_medio_s=("user_attrs_train_time", "mean"),
        epocas_medias=("user_attrs_epochs_run", "mean"),
        params_medios=("user_attrs_n_params", "mean"),
    )
    return g.round(4)


for col in ["params_optimizer", "params_activation", "params_n_layers"]:
    print(f"--- por {col.replace('params_', '')} ---")
    display(summary_by(col))

In [ ]:
def group_boxplot(ax, df, col, title, xlabel):
    cats = [c for c in sorted(df[col].dropna().unique()) if (df[col] == c).any()]
    data = [df.loc[df[col] == c, "value"].values for c in cats]
    ax.boxplot(data, showmeans=True)
    for i, d in enumerate(data, start=1):
        ax.scatter(np.random.normal(i, 0.04, len(d)), d, alpha=0.6, s=18, color="tab:blue", zorder=3)
    ax.set_xticks(range(1, len(cats) + 1))
    ax.set_xticklabels([f"{c}\n(n={len(d)})" for c, d in zip(cats, data)])
    ax.set(title=title, xlabel=xlabel, ylabel="val_acc")


fig, axes = plt.subplots(2, 3, figsize=(18, 9))
group_boxplot(axes[0, 0], done, "params_optimizer", "Rendimiento por optimizador", "optimizador")
group_boxplot(axes[0, 1], done, "params_activation", "Rendimiento por activación", "activación")
group_boxplot(axes[0, 2], done, "params_n_layers", "Rendimiento por cantidad de capas ocultas", "capas ocultas")

# val_acc vs tiempo de entrenamiento, por optimizador
colors = {"adam": "tab:blue", "rmsprop": "tab:orange", "sgd": "tab:green"}
for opt_name, g in done.groupby("params_optimizer"):
    axes[1, 0].scatter(g["user_attrs_train_time"], g["value"], label=opt_name, color=colors.get(opt_name), alpha=0.8)
axes[1, 0].set(title="Rendimiento vs tiempo de entrenamiento", xlabel="tiempo de entrenamiento (s)", ylabel="val_acc")
axes[1, 0].legend()

# val_acc vs cantidad de parámetros
for opt_name, g in done.groupby("params_optimizer"):
    axes[1, 1].scatter(g["user_attrs_n_params"], g["value"], label=opt_name, color=colors.get(opt_name), alpha=0.8)
axes[1, 1].set_xscale("log")
axes[1, 1].set(title="Rendimiento vs tamaño del modelo", xlabel="parámetros (log)", ylabel="val_acc")
axes[1, 1].legend()

# Historia de la optimización
axes[1, 2].scatter(done["number"], done["value"], alpha=0.7, label="trial completo")
if len(prun):
    axes[1, 2].scatter(prun["number"], [axes[1, 2].get_ylim()[0]] * len(prun), marker="x", color="red", label="trial podado")
axes[1, 2].plot(done["number"], done["value"].cummax(), color="k", label="mejor hasta el momento")
axes[1, 2].set(title="Historia de la optimización", xlabel="trial", ylabel="val_acc")
axes[1, 2].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Curvas de val_acc por época de todos los trials: se ve dónde actuó el pruner
fig, ax = plt.subplots(figsize=(10, 5))
for t in study_mlp.trials:
    if not t.intermediate_values:
        continue
    steps, vals = zip(*sorted(t.intermediate_values.items()))
    pruned_t = t.state.name == "PRUNED"
    ax.plot([s + 1 for s in steps], vals, color="tab:red" if pruned_t else "tab:blue",
            alpha=0.45 if pruned_t else 0.7, lw=1)
ax.plot([], [], color="tab:blue", label="completo")
ax.plot([], [], color="tab:red", label="podado")
ax.set(title="val_acc por época según estado del trial", xlabel="época", ylabel="val_acc")
ax.legend()
plt.tight_layout()
plt.show()

# Importancia de hiperparámetros
try:
    imp = optuna.importance.get_param_importances(study_mlp)
    imp = pd.Series(imp).sort_values()
    plt.figure(figsize=(8, 6))
    plt.barh(imp.index, imp.values)
    plt.title("Importancia de hiperparámetros (fANOVA)")
    plt.xlabel("importancia")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("No se pudo calcular la importancia de hiperparámetros:", e)